# Validation

**Objective:** Reject invalid data at the API boundary before business logic runs.

## Simple version

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=100)
    priority: int = Field(ge=1, le=5)


try:
    TaskCreate(title="", priority=9)
except ValidationError as error:
    print("Rejected fields:", error.error_count())

## Polished version

In [ ]:
from typing import Optional

from pydantic import BaseModel, Field, ValidationError


class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    priority: int = Field(default=3, ge=1, le=5)


class TaskUpdate(BaseModel):
    title: Optional[str] = Field(default=None, min_length=1, max_length=200)
    priority: Optional[int] = Field(default=None, ge=1, le=5)


class TaskService:
    def create(self, body: TaskCreate) -> dict:
        return {"id": 1, **body.model_dump()}

    def update(self, task: dict, body: TaskUpdate) -> dict:
        return {**task, **body.model_dump(exclude_unset=True)}


service = TaskService()
task = service.create(TaskCreate.model_validate({"title": "Validate input"}))
task = service.update(task, TaskUpdate.model_validate({"priority": 5}))

try:
    TaskCreate.model_validate({"title": "", "priority": 9})
except ValidationError as error:
    print("Rejected:", [item["loc"][0] for item in error.errors()])

print(task)